In [74]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

### 메시지발송 시간, 재방문 여부에 따른 종속변수 코딩
* sess 이후 메시지 발송 이력이 없으면 -1
* sess 이후 메시지 발송 이력이 있고, 재방문을 했다면 1
* sess 이후 메시지 발송 이력이 있고, 재방문을 하지 않았다면 0
* 메시지 데이터 merge

In [15]:
sess = pd.read_csv('./data/cv_data.csv', index_col=0)
sess['LST_SESS_TIME'] = pd.to_datetime(sess['LST_SESS_TIME'])
print(sess.shape)
sess.head(2)

(293032, 16)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,EVNT_CNT,INCS_NO,LST_SESS_TIME,AVG_DEPTH,day_off,SEX_CD,AVG_SAL_AMT,AGE_30,AGE_50,AGE_60,AGE_1020
0,000117b4880909a482cbfb7a65ddf0f71722298631,22,6,1,5,22,56c306ce1d710f24c64c604ce3dfc456c52d28ce4b62c0...,2024-07-30 09:19:43.659,26.250000,0,1,7695.454545,1,0,0,0
1,000117b4880909a482cbfb7a65ddf0f71722301833,51,13,3,23,51,56c306ce1d710f24c64c604ce3dfc456c52d28ce4b62c0...,2024-07-30 10:29:11.286,44.677419,0,1,7695.454545,1,0,0,0


In [16]:
msg = pd.read_csv('./data/msg_indicate.csv', index_col = 0)
msg['STND_YMD'] = pd.to_datetime(msg['STND_YMD'])
print(msg.shape)
msg.head(2)

(4931532, 6)


,STND_YMD,INCS_NO,msg_length,max_discount,emoji_count,time_pressure
0,2024-10-28,f3f306c20e5caa497df4bbbd4732161eb63f65b92544b3...,76,64.0,1,1
1,2024-09-08,906522474e459ca44336c858e76b7f6cc4a8b76abba594...,49,33.0,2,0


In [17]:
cust_list = sess['INCS_NO'].unique()
cust_list

array(['56c306ce1d710f24c64c604ce3dfc456c52d28ce4b62c0b57c36c48564f39c0c286947ed8802a377f9b2341d268077296952c72b330a69bc4d80d4f87615fa62',
       'e4ae210842968d7b7bd92cc620628ee81447369137348e9e6c8253a3769f746ee78b57dad8d8c6089caa0e07a38e44010bf25e4cd291649be154f199798ac20f',
       'cbfbcaec8d9089789dc079545bed286ec96c227a50ca6c17c4e9d02befb96f044a16a9bf524c4434179ebe4530e237988acf6af1d7aed9f3607d9ffd19e1f9cf',
       ...,
       'cc4b579499d587a106b5e0faebb61fdf2e65aaf5d1b5c8eae09393f7a8d087b4df86ed0e9b9ed74ff26f824f88064b1d667d8f7a1aaaf06314f2e2f91be8dfb7',
       '27caac0ccfe0bd744368a07e510eecf8452a16d2451d28283038ea856f82b21362582de18d210a8e2563c1209584574bf7d8818e26b80949b3d300b7965fcae1',
       'fcab5ec215f0a55c305f12f44066269f0e5b346756ba66022ca2ab2934f38173cb1686e26d3a952ffeb87f417996b166fb75d483ebdfcdee58e076ba3cd9a5ff'],
      dtype=object)

In [75]:
new_msg = msg[msg['INCS_NO'].isin(cust_list)]
new_msg['msg_index'] = new_msg.index
new_msg = new_msg[(new_msg['STND_YMD'] >= pd.to_datetime('2024-06-01')) & (new_msg['STND_YMD'] < pd.to_datetime('2024-08-01'))]
new_msg

,STND_YMD,INCS_NO,msg_length,max_discount,emoji_count,time_pressure,msg_index
2,2024-06-21,a5028798964bfd1c9e1077155642b7196abca1d43d6ab6...,30,32.000000,0,0,2
26,2024-07-29,684734b317ac76a35202397a4cd6ecfbc23cfa38274e76...,273,40.000000,1,0,26
27,2024-07-29,684734b317ac76a35202397a4cd6ecfbc23cfa38274e76...,329,40.000000,1,0,27
28,2024-07-29,684734b317ac76a35202397a4cd6ecfbc23cfa38274e76...,291,40.000000,1,0,28
29,2024-07-29,684734b317ac76a35202397a4cd6ecfbc23cfa38274e76...,295,40.000000,1,0,29
...,...,...,...,...,...,...,...
5463200,2024-06-14,b491aa1ad5e874ae610c7931b40949c035cc6b4a9ee957...,40,50.000000,0,0,5463200
5463220,2024-07-12,380f002133125c673524d9eb310de1948ff5342a9c8fc4...,62,33.000000,0,0,5463220
5463229,2024-07-12,bbcc097782b6072d13605333820de38aa8f11fc9c8699a...,20,50.000000,0,0,5463229
5463239,2024-07-26,4dc13ea1a4a1613d8175252a0a3ae3794611f642bc6775...,62,33.333333,1,0,5463239


In [76]:
#gpt
new_msg["DATE_TIME"] = new_msg["STND_YMD"] + pd.Timedelta(hours=10)
new_msg.reset_index()
new_msg

,STND_YMD,INCS_NO,msg_length,max_discount,emoji_count,time_pressure,msg_index,DATE_TIME
2,2024-06-21,a5028798964bfd1c9e1077155642b7196abca1d43d6ab6...,30,32.000000,0,0,2,2024-06-21 10:00:00
26,2024-07-29,684734b317ac76a35202397a4cd6ecfbc23cfa38274e76...,273,40.000000,1,0,26,2024-07-29 10:00:00
27,2024-07-29,684734b317ac76a35202397a4cd6ecfbc23cfa38274e76...,329,40.000000,1,0,27,2024-07-29 10:00:00
28,2024-07-29,684734b317ac76a35202397a4cd6ecfbc23cfa38274e76...,291,40.000000,1,0,28,2024-07-29 10:00:00
29,2024-07-29,684734b317ac76a35202397a4cd6ecfbc23cfa38274e76...,295,40.000000,1,0,29,2024-07-29 10:00:00
...,...,...,...,...,...,...,...,...
5463200,2024-06-14,b491aa1ad5e874ae610c7931b40949c035cc6b4a9ee957...,40,50.000000,0,0,5463200,2024-06-14 10:00:00
5463220,2024-07-12,380f002133125c673524d9eb310de1948ff5342a9c8fc4...,62,33.000000,0,0,5463220,2024-07-12 10:00:00
5463229,2024-07-12,bbcc097782b6072d13605333820de38aa8f11fc9c8699a...,20,50.000000,0,0,5463229,2024-07-12 10:00:00
5463239,2024-07-26,4dc13ea1a4a1613d8175252a0a3ae3794611f642bc6775...,62,33.333333,1,0,5463239,2024-07-26 10:00:00


In [77]:
merged_df = pd.merge(
    new_msg[["msg_index","INCS_NO", "DATE_TIME"]],
    sess[["INCS_NO", "LST_SESS_TIME", "SESS_ID"]],
    on='INCS_NO',
    how='left'
)

In [78]:
# 차이를 일 단위로 계산
merged_df['date_diff'] = (merged_df['DATE_TIME'] - merged_df['LST_SESS_TIME']).dt.total_seconds() / (24 * 3600)
# -2일에서 2일 사이의 범위로 필터링
test_df = merged_df[merged_df['date_diff'] <= 1]
test_df.head(2)

,msg_index,INCS_NO,DATE_TIME,LST_SESS_TIME,SESS_ID,date_diff
0,2,a5028798964bfd1c9e1077155642b7196abca1d43d6ab6...,2024-06-21 10:00:00,2024-06-28 19:28:21.829,b689c491e744a66411c3d5ea1c574c5e1719570370,-7.394697
16,47,72db74633d53d603d376905eb946ef7448e4c5e801460e...,2024-07-12 10:00:00,2024-07-14 09:48:56.255,97A3754708664784B4F45C853C9E6ACC1720917359,-1.992318


In [79]:
def dv_coding(date_diff):
    if date_diff > 0:
        return 1
    else:
        return 0

test_df['y'] = test_df['date_diff'].apply(dv_coding)

In [80]:
test_df.y.value_counts()

y
0    1016133
1      31903
Name: count, dtype: int64

### new_msg, test_df, sess, cust 데이터 결합

In [83]:
final_df = test_df[['msg_index', 'INCS_NO', 'SESS_ID', 'y']]
final_df.head(2)

,msg_index,INCS_NO,SESS_ID,y
0,2,a5028798964bfd1c9e1077155642b7196abca1d43d6ab6...,b689c491e744a66411c3d5ea1c574c5e1719570370,0
16,47,72db74633d53d603d376905eb946ef7448e4c5e801460e...,97A3754708664784B4F45C853C9E6ACC1720917359,0
